### สมาชิกกลุ่ม

| ชื่อ-นามสกุล | รหัสนักศึกษา |
| :--- | :--- |
| นางสาวณภัทร วานิชวัตกากร | `67070226` |
| นายพลาธิป เหมวุฒิ | `67070255` |
| นายอองรักษ์ วณิชชานัย | `67070297` |
| นายเอื้ออังกูร ชัยวิวัฒน์พร | `67070302` |

# 📊 Phase 1: Sentiment Labeling — Web Scraping Data
### Classical Approach: TF-IDF + Machine Learning

**เป้าหมาย:** Train โมเดลจาก `Financial_Sentiment_Analysis_1.csv` แล้วนำมา predict label ให้กับ `clean_text_(Web_Scraping).csv (ให้โมเดลเรียนรู้ pattern จาก Financial แล้วนำมา predict ให้ Web Scraping)`

**Pipeline:**
1. Load & Explore ข้อมูล
2. Preprocessing
3. Train โมเดล (TF-IDF + Logistic Regression)
4. Evaluate โมเดล (F1, Precision, Recall)
5. Predict label ให้ Web Scraping data
6. Export ไฟล์ CSV พร้อม label

## 📦 Cell 1: Import Libraries

In [84]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')

✅ Libraries imported successfully


## 📂 Cell 2: Load Data

In [85]:
# โหลดข้อมูล Training (Financial Sentiment)
df_train = pd.read_csv('Financial Sentiment Analysis 1.csv')

# โหลดข้อมูล Web Scraping ที่ต้องการ label
df_web = pd.read_csv('clean_text_(Web_Scraping).csv')

print('=== Financial Sentiment Dataset (Training) ===')
print(f'Shape: {df_train.shape}')
print(df_train.head(3))
print()
print('=== Web Scraping Dataset (To be labeled) ===')
print(f'Shape: {df_web.shape}')
print(df_web.head(3))

=== Financial Sentiment Dataset (Training) ===
Shape: (5842, 2)
                                            Sentence Sentiment
0  The GeoSolutions technology will leverage Bene...  positive
1  $ESI on lows, down $1.50 to $2.50 BK a real po...  negative
2  For the last quarter of 2010 , Componenta 's n...  positive

=== Web Scraping Dataset (To be labeled) ===
Shape: (839, 1)
                                            Sentence
0  Three Top Dividend Stocks To Consider. As Febr...
1  Dow Jones Futures Fall After S&P 500 Hits Resi...
2  Here's Why I Wouldn't Touch Medical Properties...


## 📊 Cell 3: Exploratory Data Analysis (EDA)

In [86]:
# การกระจายของ Sentiment ใน Training data
print('=== Sentiment Distribution (Training Data) ===')
print(df_train['Sentiment'].value_counts())
print()
print(f'Class balance ratio:')
print(df_train['Sentiment'].value_counts(normalize=True).round(3))

=== Sentiment Distribution (Training Data) ===
Sentiment
neutral     3130
positive    1852
negative     860
Name: count, dtype: int64

Class balance ratio:
Sentiment
neutral     0.536
positive    0.317
negative    0.147
Name: proportion, dtype: float64


## 🧹 Cell 4: Text Preprocessing

In [87]:
def clean_text(text):
    """
    Preprocessing Pipeline:
    1. แปลงเป็น lowercase
    2. ลบ HTML tags
    3. ลบ URLs
    4. ลบ Emojis และ special characters
    5. ลบ extra whitespace
    """
    if not isinstance(text, str):
        return ''
    
    # Lowercase
    text = text.lower()
    
    # ลบ HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # ลบ URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # ลบ Emojis (Unicode ranges)
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text, flags=re.UNICODE)
    
    # ลบ special characters เก็บไว้แค่ตัวอักษร ตัวเลข และ space
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # ลบ extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
df_train['clean_text'] = df_train['Sentence'].apply(clean_text)
df_web['clean_text'] = df_web['Sentence'].apply(clean_text)

# ตรวจสอบผล
print('=== ตัวอย่าง Preprocessing ===')
print('Original:', df_train['Sentence'].iloc[0][:100])
print('Cleaned :', df_train['clean_text'].iloc[0][:100])
print()

# ลบ rows ที่ clean_text ว่าง
df_train = df_train[df_train['clean_text'].str.len() > 5].reset_index(drop=True)
df_web = df_web[df_web['clean_text'].str.len() > 5].reset_index(drop=True)

print(f'Training data after cleaning: {df_train.shape[0]} rows')
print(f'Web Scraping data after cleaning: {df_web.shape[0]} rows')
print('✅ Preprocessing complete')

=== ตัวอย่าง Preprocessing ===
Original: The GeoSolutions technology will leverage Benefon 's GPS solutions by providing Location Based Searc
Cleaned : the geosolutions technology will leverage benefon s gps solutions by providing location based search

Training data after cleaning: 5842 rows
Web Scraping data after cleaning: 839 rows
✅ Preprocessing complete


## 🔢 Cell 5: TF-IDF Vectorization

In [88]:
# แบ่ง Train/Test split
X = df_train['clean_text']
y = df_train['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print()
print('Train label distribution:')
print(y_train.value_counts())

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,      # Top 10,000 features
    ngram_range=(1, 2),      # Unigram + Bigram
    min_df=2,                # ต้องปรากฏอย่างน้อย 2 ครั้ง
    max_df=0.95,             # ตัดคำที่ปรากฏมากกว่า 95% ของ documents
    sublinear_tf=True        # ใช้ log scaling สำหรับ TF
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'\nTF-IDF Matrix shape (Train): {X_train_tfidf.shape}')
print(f'TF-IDF Matrix shape (Test):  {X_test_tfidf.shape}')
print('✅ TF-IDF Vectorization complete')

Train size: 4673 | Test size: 1169

Train label distribution:
Sentiment
neutral     2504
positive    1481
negative     688
Name: count, dtype: int64

TF-IDF Matrix shape (Train): (4673, 10000)
TF-IDF Matrix shape (Test):  (1169, 10000)
✅ TF-IDF Vectorization complete


## 🤖 Cell 6: Train & Compare 3 Classical Models

In [89]:
# กำหนดโมเดลที่จะเปรียบเทียบ
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, C=1.0),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'f1_weighted': f1
    }
    print(f'  → F1 Score (weighted): {f1:.4f}')
    print()

# หา Best Model
best_model_name = max(results, key=lambda x: results[x]['f1_weighted'])
print(f'🏆 Best Model: {best_model_name} (F1 = {results[best_model_name]["f1_weighted"]:.4f})')

Training Logistic Regression...
  → F1 Score (weighted): 0.6775

Training Naive Bayes...
  → F1 Score (weighted): 0.6912

Training Random Forest...
  → F1 Score (weighted): 0.6176

🏆 Best Model: Naive Bayes (F1 = 0.6912)


## 📈 Cell 7: Detailed Evaluation — Best Model

In [90]:
best_model = results[best_model_name]['model']
y_pred_best = results[best_model_name]['predictions']

print(f'=== Evaluation Report: {best_model_name} ===')
print(classification_report(y_test, y_pred_best, digits=4))

=== Evaluation Report: Naive Bayes ===
              precision    recall  f1-score   support

    negative     0.3333    0.3023    0.3171       172
     neutral     0.7492    0.7923    0.7702       626
    positive     0.7521    0.7116    0.7313       371

    accuracy                         0.6946      1169
   macro avg     0.6116    0.6021    0.6062      1169
weighted avg     0.6890    0.6946    0.6912      1169



## 🔍 Cell 8: Error Analysis

In [91]:
# สร้าง DataFrame สำหรับ Error Analysis
test_df = X_test.to_frame().copy()
test_df['actual'] = y_test.values
test_df['predicted'] = y_pred_best
test_df['correct'] = test_df['actual'] == test_df['predicted']

# แสดงตัวอย่างที่ predict ผิด
errors = test_df[~test_df['correct']]
print(f'=== Error Analysis ===')
print(f'Total errors: {len(errors)} / {len(test_df)} ({len(errors)/len(test_df)*100:.1f}%)')
print()

# Error pattern
print('Error patterns (Actual → Predicted):')
error_patterns = errors.groupby(['actual', 'predicted']).size().reset_index(name='count')
error_patterns = error_patterns.sort_values('count', ascending=False)
print(error_patterns.to_string(index=False))
print()

# ตัวอย่าง error cases
print('=== ตัวอย่างประโยคที่ predict ผิด ===')
for _, row in errors.head(5).iterrows():
    print(f'Text     : {row["clean_text"][:80]}...')
    print(f'Actual   : {row["actual"]}  |  Predicted: {row["predicted"]}')
    print('-' * 60)

=== Error Analysis ===
Total errors: 357 / 1169 (30.5%)

Error patterns (Actual → Predicted):
  actual predicted  count
positive   neutral     89
 neutral  negative     86
negative   neutral     77
 neutral  positive     44
negative  positive     43
positive  negative     18

=== ตัวอย่างประโยคที่ predict ผิด ===
Text     : nordic business report 26 june 2006 metso corporation wins eur50m equipment orde...
Actual   : positive  |  Predicted: neutral
------------------------------------------------------------
Text     : rad all my charts are flashing oversold...
Actual   : neutral  |  Predicted: positive
------------------------------------------------------------
Text     : operating loss landed at eur39m including one offs and at eur27m excluding one o...
Actual   : neutral  |  Predicted: negative
------------------------------------------------------------
Text     : the liquidity providing was interrupted on may 11 2007 when aspocomp group oyj s...
Actual   : negative  |  Predicted:

## 🎯 Cell 9: Predict Labels for Web Scraping Data

In [92]:
# Transform Web Scraping data ด้วย TF-IDF ที่ train แล้ว
X_web_tfidf = tfidf.transform(df_web['clean_text'])

# Predict labels
df_web['Sentiment'] = best_model.predict(X_web_tfidf)

# Predict probability (confidence score)
if hasattr(best_model, 'predict_proba'):
    proba = best_model.predict_proba(X_web_tfidf)
    df_web['confidence'] = proba.max(axis=1).round(4)
    classes = best_model.classes_
    for i, cls in enumerate(classes):
        df_web[f'prob_{cls}'] = proba[:, i].round(4)

print('=== Predicted Sentiment Distribution (Web Scraping) ===')
print(df_web['Sentiment'].value_counts())
print()
print('========== ตัวอย่างข้อมูล ==========')

# แสดงผลเป็นตาราง DataFrame style
preview = pd.concat([
    df_output[df_output['Sentiment'] == s].head(3)
    for s in ['positive', 'neutral', 'negative']
]).reset_index(drop=True)

# ตัดข้อความให้พอดี
preview['Sentence'] = preview['Sentence'].str[:80] + '...'

# แสดงผลแบบมีสี
def highlight_sentiment(val):
    colors = {'positive': 'background-color: #d4edda; color: #155724',
              'neutral':  'background-color: #cce5ff; color: #004085',
              'negative': 'background-color: #f8d7da; color: #721c24'}
    return colors.get(val, '')

preview.style \
    .applymap(highlight_sentiment, subset=['Sentiment']) \
    .set_properties(**{'text-align': 'left', 'font-size': '13px'}) \
    .set_table_styles([{'selector': 'th', 'props': [('font-size', '13px'),
                                                     ('background-color', '#343a40'),
                                                     ('color', 'white'),
                                                     ('text-align', 'left')]}])

=== Predicted Sentiment Distribution (Web Scraping) ===
Sentiment
neutral     507
positive    294
negative     38
Name: count, dtype: int64

========== ตัวอย่างข้อมูล ==========


,Sentence,Sentiment
0,"Three Top Dividend Stocks To Consider. As February begins, U.S. stock indexes ha...",positive
1,"Tech Qualms, Walmart Outlook Damp Wall Street Pre-Bell; Asia Up, Europe Off. Wal...",positive
2,Does Index Upgrades for Ciena CIEN Change The Bull Case For Its AI-Optical Story...,positive
3,Here's Why I Wouldn't Touch Medical Properties Trust With a 10 Foot Pole. Medica...,neutral
4,Hawkish hints in Fed minutes; Walmart to report - what s moving markets. Investi...,neutral
5,"Zacks.com featured highlights include Harmony Biosciences, Tripadvisor, The AES ...",neutral
6,"Dow Jones Futures Fall After S&P 500 Hits Resistance; Carvana Dives, Walmart Due...",negative
7,Markets Dropping. Oil Prices Surge Amid U.S.-Iran Fears.U.S. markets were headed...,negative
8,FTSE 100 LIVE: Stocks slump as miners and British Gas owner Centrica pull London...,negative


## 💾 Cell 10: Export Labeled CSV

In [93]:
# เลือก columns ที่ต้องการ export
df_output = df_web[['Sentence', 'Sentiment']].copy()

# Save
output_path = 'web_scraping_labeled.csv'
df_output.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ Exported: {output_path}')
print(f'Total rows: {len(df_output)}')
print()
print('=== Preview (5 rows per sentiment) ===')
for sentiment in ['positive', 'neutral', 'negative']:
    subset = df_output[df_output['Sentiment'] == sentiment]
    print(f'\n--- {sentiment.upper()} ({len(subset)} rows) ---')
    print(subset.head(2)[['Sentence', 'Sentiment']].to_string(index=False))

✅ Exported: web_scraping_labeled.csv
Total rows: 839

=== Preview (5 rows per sentiment) ===

--- POSITIVE (294 rows) ---
                                                                                                                                                                                                                                                                                                                                                                                               Sentence Sentiment
Three Top Dividend Stocks To Consider. As February begins, U.S. stock indexes have shown strength, with the Dow Jones Industrial Average adding 515 points and the S&P 500 nearing a record high. In this dynamic market environment, dividend stocks can offer stability and income potential, making them an appealing consideration for investors seeking to balance growth with consistent returns.  positive
                                                                          

## 🔗 Cell 11: รวม Web Scraping + Financial CSV

In [94]:
# โหลดไฟล์ทั้งสอง
df_financial = pd.read_csv('Financial Sentiment Analysis 1.csv')
df_labeled   = pd.read_csv('web_scraping_labeled.csv')

# รวมทั้งสองไฟล์
df_combined = pd.concat([df_financial, df_labeled], ignore_index=True)

# ตรวจสอบผล
print('=== ผลการรวมข้อมูล ===')
print(f'Financial CSV    : {len(df_financial):,} rows')
print(f'Web Scraping CSV : {len(df_labeled):,} rows')
print(f'รวมทั้งหมด       : {len(df_combined):,} rows')
print()
print('=== Sentiment Distribution (รวม) ===')
print(df_combined['Sentiment'].value_counts())
print()
print('========== ตัวอย่างข้อมูล ==========')

# แสดงผลเป็นตาราง DataFrame style
preview_combined = pd.concat([
    df_combined[df_combined['Sentiment'] == s].head(2)
    for s in ['positive', 'neutral', 'negative']
]).reset_index(drop=True)

preview_combined['Sentence'] = preview_combined['Sentence'].str[:80] + '...'

def highlight_sentiment(val):
    colors = {'positive': 'background-color: #d4edda; color: #155724',
              'neutral':  'background-color: #cce5ff; color: #004085',
              'negative': 'background-color: #f8d7da; color: #721c24'}
    return colors.get(val, '')

display(preview_combined.style \
    .applymap(highlight_sentiment, subset=['Sentiment']) \
    .set_properties(**{'text-align': 'left', 'font-size': '13px'}) \
    .set_table_styles([{'selector': 'th', 'props': [('font-size', '13px'),
                                                     ('background-color', '#343a40'),
                                                     ('color', 'white'),
                                                     ('text-align', 'left')]}]))

# Export
df_combined.to_csv('complete_dataset(Labeling).csv', index=False, encoding='utf-8-sig')
print()
print('✅ Exported: complete_dataset(Labeling).csv')

=== ผลการรวมข้อมูล ===
Financial CSV    : 5,842 rows
Web Scraping CSV : 839 rows
รวมทั้งหมด       : 6,681 rows

=== Sentiment Distribution (รวม) ===
Sentiment
neutral     3637
positive    2146
negative     898
Name: count, dtype: int64

========== ตัวอย่างข้อมูล ==========


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Benefon 's GPS solutions by providing ...,positive
1,"For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from E...",positive
2,"According to the Finnish-Russian Chamber of Commerce , all the major constructio...",neutral
3,"The Swedish buyout firm has sold its remaining 22.4 percent stake , almost eight...",neutral
4,"$ESI on lows, down $1.50 to $2.50 BK a real possibility...",negative
5,Shell's $70 Billion BG Deal Meets Shareholder Skepticism...,negative



✅ Exported: complete_dataset(Labeling).csv
